In [ ]:
!pip install streamlit transformers torch torchvision pillow --quiet

In [ ]:
%%writefile model.py
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def generate_caption(image):
    inputs = processor(images=image, return_tensors="pt").to(device)

    output = model.generate(
        **inputs,
        num_beams=7,          # better search
        max_length=60,        # longer output
        min_length=15,        # avoid short captions
        repetition_penalty=1.2,
        length_penalty=1.0,
        early_stopping=True
    )

    caption = processor.decode(output[0], skip_special_tokens=True)

    return caption


Overwriting model.py


In [ ]:
import streamlit as st
from PIL import Image
from model import generate_caption

st.set_page_config(page_title="Image Caption Generator", page_icon="🖼️")

st.title("🖼️ Image Caption Generator (BLIP Model)")
st.write("Upload an image or capture using your camera.")

# Upload option
uploaded_file = st.file_uploader(
    "Upload an Image",
    type=["jpg", "jpeg", "png"]
)

# Camera option
camera_image = st.camera_input("Or Take a Picture")

# Determine which image to use
image = None

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert("RGB")

elif camera_image is not None:
    image = Image.open(camera_image).convert("RGB")

# If we have an image
if image is not None:
    st.image(image, caption="Selected Image", width=500)

    if st.button("Generate Caption"):
        with st.spinner("Generating caption..."):
            caption = generate_caption(image)
        st.success("Caption: " + caption)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 121856 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.2.0) over (2026.2.0) ...
Setting up cloudflared (2026.2.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!streamlit run app.py &>/dev/null &

In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-02-17T11:06:57Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-02-17T11:06:57Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-02-17T11:07:00Z INF +--------------------------------------------------------------------------------------------+
2026-02-17T11:07:00Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-02-17T11:07:00Z INF |  https://tennis-made-necklace-enjoy.trycloudflare.com 